第七课

Use AI (机器学习 / 深度学习: 随机森林、Gradient Boosting Regressor、长短期记忆网络) to predict future cash flows

Use DCF -> 估值

In [25]:
import rqdatac as rq
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import RFECV
from sklearn.metrics import r2_score, mean_absolute_error

# Log in to rqdatac
rq.init()

# Get 沪深300 constituent stocks
constituents = rq.index_components('000300.XSHG')

start_date = '2010-01-01'
end_date = '2025-12-31'

factors = [
    # revenue
    'revenue',
    'operating_revenue',
    'net_operating_revenue',
    
    # valuation
    'book_value_per_share',
    'book_value_per_share_lf',
    'book_value_per_share_ttm',

    # profitability
    'return_on_equity',
    'return_on_asset',
    "profit_before_tax",
    "minority_profit",
    "net_profit",
    "net_profit_parent_company",
    "ebit",

    # leverage/liquidity
    'debt_to_equity',
    'current_ratio',
    
    # growth
    'revenue_growth_rate',
    'net_profit_growth_rate',

    # free cash flow
    'free_cash_flow_company_per_share',
    'free_cash_flow_company_per_share_ttm',

    # Balance
    "cash_equivalent", 
    "financial_asset_held_for_trading", 
    "current_assets", 
    "bill_receivable", 
    "net_accts_receivable", 
    "other_accts_receivable", 
    "inventory", 
    "deferred_expense", 
    "other_current_assets", 
    "long_term_equity_investment", 
    "net_long_term_equity_investment", 
    "real_estate_investment", 
    "net_fixed_assets", 
    "construction_in_progress", 
    "fixed_asset_to_be_disposed", 
    "intangible_assets", 
    "long_term_deferred_expenses", 
    "deferred_income_tax_assets", 
    "other_non_current_assets", 
    "short_term_loans", 
    "notes_payable", 
    "accts_payable", 
    "long_term_liabilities_due_one_year", 
    "non_current_liability_due_one_year", 
    "other_current_liabilities", 
    "current_liabilities", 
    "long_term_loans", 
    "bond_payable", 
    "long_term_payable", 
    "deferred_income_tax_liabilities", 
    "deferred_revenue", 
    "other_non_current_liabilities", 
    "total_assets", 
    "minority_interest", 
    "equity_parent_company", 
    "total_equity",
]

# Fetch financial statement ratio information
financial_data = rq.get_factor(constituents, factors, start_date=start_date, end_date=end_date)

financial_data = financial_data.fillna(0, inplace=False)

print("financial data:")
print(financial_data.head(5))

/Users/tylerpruitt/Desktop/量化投资交易/Quant Trading Desk/.venv/lib/python3.8/site-packages/rqdatac/client.py:224: UserWarning: rqdatac is already inited. Settings will be changed.
  warnings.warn("rqdatac is already inited. Settings will be changed.", stacklevel=0)


financial data:
                               revenue  operating_revenue  \
order_book_id date                                          
600803.XSHG   2010-01-04  5.576125e+08       5.576125e+08   
              2010-01-05  5.576125e+08       5.576125e+08   
              2010-01-06  5.576125e+08       5.576125e+08   
              2010-01-07  5.576125e+08       5.576125e+08   
              2010-01-08  5.576125e+08       5.576125e+08   

                          net_operating_revenue  book_value_per_share  \
order_book_id date                                                      
600803.XSHG   2010-01-04                    0.0                  1.62   
              2010-01-05                    0.0                  1.62   
              2010-01-06                    0.0                  1.62   
              2010-01-07                    0.0                  1.62   
              2010-01-08                    0.0                  1.62   

                          book_value_per_sha

In [26]:
X = financial_data

X = X.groupby([
    pd.Grouper(level='order_book_id'),
    pd.Grouper(level='date', freq='Y')
]).last().fillna(0)

X = X.reorder_levels(['order_book_id', 'date']).shift(-1)

print(X.head(5))

                               revenue  operating_revenue  \
order_book_id date                                          
000001.XSHE   2010-12-31  2.070140e+10       2.070140e+10   
              2011-12-31  2.953167e+10       2.953167e+10   
              2012-12-31  3.734500e+10       3.734500e+10   
              2013-12-31  5.465100e+10       5.465100e+10   
              2014-12-31  7.115200e+10       7.115200e+10   

                          net_operating_revenue  book_value_per_share  \
order_book_id date                                                      
000001.XSHE   2010-12-31                    0.0                 13.61   
              2011-12-31                    0.0                 15.95   
              2012-12-31                    0.0                 11.58   
              2013-12-31                    0.0                 11.09   
              2014-12-31                    0.0                 10.98   

                          book_value_per_share_lf  book_valu

In [27]:
# Fetch next year's free cash flow
next_year_free_cash_flow = rq.get_factor(constituents, 'free_cash_flow_company_per_share_ttm', start_date='2011-01-01', end_date=end_date)

next_year_free_cash_flow = next_year_free_cash_flow.groupby([
    pd.Grouper(level='order_book_id'),
    pd.Grouper(level='date', freq='Y')
]).last().fillna(0)

next_year_free_cash_flow = next_year_free_cash_flow.reorder_levels(['order_book_id', 'date']).shift(-1)

next_year_free_cash_flow = next_year_free_cash_flow.fillna(0, inplace=False)

y = next_year_free_cash_flow

print(next_year_free_cash_flow)

                          free_cash_flow_company_per_share_ttm
order_book_id date                                            
000001.XSHE   2011-12-31                              4.734325
              2012-12-31                              4.077166
              2013-12-31                              2.126234
              2014-12-31                              1.859431
              2015-12-31                              1.384475
...                                                        ...
688981.XSHG   2021-12-31                              1.428541
              2022-12-31                              1.360557
              2023-12-31                              0.078630
              2024-12-31                             -0.462069
              2025-12-31                              0.000000

[3712 rows x 1 columns]


In [28]:
# Remove 2025 data because 2026 has not finished yet
X = X[X.index.get_level_values('date') != end_date]
print(X)

                               revenue  operating_revenue  \
order_book_id date                                          
000001.XSHE   2010-12-31  2.070140e+10       2.070140e+10   
              2011-12-31  2.953167e+10       2.953167e+10   
              2012-12-31  3.734500e+10       3.734500e+10   
              2013-12-31  5.465100e+10       5.465100e+10   
              2014-12-31  7.115200e+10       7.115200e+10   
...                                ...                ...   
688981.XSHG   2020-12-31  2.537100e+10       2.537100e+10   
              2021-12-31  3.776356e+10       3.776356e+10   
              2022-12-31  3.309824e+10       3.309824e+10   
              2023-12-31  4.187872e+10       4.187872e+10   
              2024-12-31  4.951042e+10       4.951042e+10   

                          net_operating_revenue  book_value_per_share  \
order_book_id date                                                      
000001.XSHE   2010-12-31                    0.0             

In [29]:
# Assuming your data is already multi-indexed with (order_book_id, date)
# Align features (year t) with target (year t+1)

# Method 1: Shift per stock
def align_features_target(X, y):
    """
    X: features at time t
    y: target at time t
    Returns: X_aligned (time t), y_aligned (time t+1)
    """
    # Reset index to work with dates
    X_df = X.reset_index()
    y_df = y.reset_index()
    
    # Rename target column for clarity
    y_df = y_df.rename(columns={'free_cash_flow_company_per_share_ttm': 'target'})
    
    # Create a shifted version of y for each stock
    aligned_pairs = []
    
    for stock in X_df['order_book_id'].unique():
        X_stock = X_df[X_df['order_book_id'] == stock].copy()
        y_stock = y_df[y_df['order_book_id'] == stock].copy()
        
        # Sort by date
        X_stock = X_stock.sort_values('date')
        y_stock = y_stock.sort_values('date')
        
        # Shift y backwards by 1 year (so y at t-1 pairs with X at t)
        # Or more intuitively: align X at year t with y at year t+1
        y_stock['target_next_year'] = y_stock['target'].shift(-1)
        
        # Merge on date
        merged = X_stock.merge(
            y_stock[['order_book_id', 'date', 'target_next_year']], 
            on=['order_book_id', 'date']
        )
        
        # Drop rows where next year's target is missing
        merged = merged.dropna(subset=['target_next_year'])
        
        aligned_pairs.append(merged)
    
    # Combine all stocks
    aligned_df = pd.concat(aligned_pairs, ignore_index=True)
    
    # Separate X and y
    X_aligned = aligned_df.drop(['order_book_id', 'date', 'target_next_year'], axis=1)
    y_aligned = aligned_df['target_next_year']
    
    return X_aligned, y_aligned, aligned_df[['order_book_id', 'date']]  # Keep metadata

def robust_align_features_target(X, y):
    """
    Robust alignment handling index mismatches
    """
    # Reset indices to columns
    X_df = X.reset_index()
    y_df = y.reset_index()
    
    # Rename target column
    y_df = y_df.rename(columns={'free_cash_flow_company_per_share_ttm': 'target'})
    
    # Sort both by stock and date
    X_df = X_df.sort_values(['order_book_id', 'date'])
    y_df = y_df.sort_values(['order_book_id', 'date'])
    
    aligned_pairs = []
    
    for stock in X_df['order_book_id'].unique():
        X_stock = X_df[X_df['order_book_id'] == stock].copy()
        y_stock = y_df[y_df['order_book_id'] == stock].copy()
        
        if len(y_stock) == 0:
            print(f"Warning: No target data for {stock}")
            continue
        
        # Ensure both have consistent date ranges
        common_dates = set(X_stock['date']).intersection(set(y_stock['date']))
        if len(common_dates) == 0:
            print(f"Warning: No common dates for {stock}")
            continue
        
        X_stock = X_stock[X_stock['date'].isin(common_dates)]
        y_stock = y_stock[y_stock['date'].isin(common_dates)]
        
        # Sort by date
        X_stock = X_stock.sort_values('date')
        y_stock = y_stock.sort_values('date')
        
        # Shift target to next year
        y_stock['target_next_year'] = y_stock['target'].shift(-1)
        
        # Merge on date
        merged = X_stock.merge(
            y_stock[['order_book_id', 'date', 'target_next_year']],
            on=['order_book_id', 'date'],
            how='inner'
        )
        
        # Drop rows with NaN target (last year of each stock)
        merged = merged.dropna(subset=['target_next_year'])
        
        if len(merged) > 0:
            aligned_pairs.append(merged)
    
    if len(aligned_pairs) == 0:
        raise ValueError("No aligned data found. Check your date ranges.")
    
    # Combine all stocks
    aligned_df = pd.concat(aligned_pairs, ignore_index=True)
    
    # Separate X, y, and metadata
    feature_cols = [col for col in X_df.columns if col not in ['order_book_id', 'date']]
    X_aligned = aligned_df[feature_cols]
    y_aligned = aligned_df['target_next_year']
    metadata = aligned_df[['order_book_id', 'date']]
    
    return X_aligned, y_aligned, metadata

# Apply the robust alignment
try:
    X_aligned, y_aligned, metadata = robust_align_features_target(X, y)
    print(f"Success! Aligned {len(X_aligned)} samples")
    print(f"X shape: {X_aligned.shape}")
    print(f"y shape: {y_aligned.shape}")
    print(f"Date range: {metadata['date'].min()} to {metadata['date'].max()}")
except ValueError as e:
    print(f"Alignment failed: {e}")

# # Apply alignment
# X_aligned, y_aligned, metadata = align_features_target(X, y)

# print(f"Original X shape: {X.shape}")
# print(f"Original y shape: {y.shape}")
# print(f"Aligned X shape: {X_aligned.shape}")
# print(f"Aligned y shape: {y_aligned.shape}")

Success! Aligned 3113 samples
X shape: (3113, 55)
y shape: (3113,)
Date range: 2011-12-31 00:00:00 to 2023-12-31 00:00:00


In [30]:
print(X_aligned.head(20))

         revenue  operating_revenue  net_operating_revenue  \
0   2.953167e+10       2.953167e+10                    0.0   
1   3.734500e+10       3.734500e+10                    0.0   
2   5.465100e+10       5.465100e+10                    0.0   
3   7.115200e+10       7.115200e+10                    0.0   
4   8.196800e+10       8.196800e+10                    0.0   
5   7.983300e+10       7.983300e+10                    0.0   
6   8.666400e+10       8.666400e+10                    0.0   
7   1.029580e+11       1.029580e+11                    0.0   
8   1.165640e+11       1.165640e+11                    0.0   
9   1.271900e+11       1.271900e+11                    0.0   
10  1.382650e+11       1.382650e+11                    0.0   
11  1.276340e+11       1.276340e+11                    0.0   
12  1.115820e+11       1.115820e+11                    0.0   
13  4.612822e+10       4.612822e+10                    0.0   
14  6.341532e+10       6.341532e+10                    0.0   
15  6.31

In [31]:
print(y_aligned.head(20))

0     4.077166
1     2.126234
2     1.859431
3     1.384475
4     0.981718
5     1.520333
6     0.805708
7     2.159187
8     2.228611
9     1.980620
10    3.052007
11    1.533604
12    2.002559
13    0.542117
14    1.594830
15    1.217993
16    2.933680
17    1.477566
18    3.296967
19    3.636659
Name: target_next_year, dtype: float64


## 随机森林

In [32]:
## 随机森林模型

# Train test split 80/20
split = int(len(X_aligned) * 0.8)

X_train, X_test = X_aligned.iloc[:split], X_aligned.iloc[split:]
y_train, y_test = y_aligned.iloc[:split], y_aligned.iloc[split:]

# Define hyperparameters once
rf_params = dict(
    n_estimators=100,
    max_depth=3,
    min_samples_leaf=5,
    max_features='sqrt'
)

# Feature selection
model = RandomForestRegressor(**rf_params)
selector = RFECV(estimator=model, cv=5)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)  # apply same selection to test set

# Cross-validate, then final fit
rf_model = RandomForestRegressor(**rf_params)
cv_scores = cross_val_score(rf_model, X_train_selected, y_train, cv=5, scoring='r2')
print(f"Cross-validation R^2 scores: {cv_scores}")

# Fit the model
rf_model.fit(X_train_selected, y_train)

# Evaluate
y_pred = rf_model.predict(X_test_selected)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f"Test R^2: {r2}, Test MAE: {mae}")

Cross-validation R^2 scores: [0.55658187 0.36819379 0.53113096 0.28218718 0.42260819]
Test R^2: 0.2987120282156053, Test MAE: 0.7520846268525686


## Gradient Boosting Regression Model

In [33]:
# Grading Boosting Regression Model

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score

# Train-test split
split = int(len(X_aligned) * 0.8)

X_train, X_test = X_aligned.iloc[:split], X_aligned.iloc[split:]
y_train, y_test = y_aligned.iloc[:split], y_aligned.iloc[split:]

# Gradient Boosting Regressor model
gbr_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

# Cross validation
gbr_cv_scores = cross_val_score(
    gbr_model,
    X_train,
    y_train.values.ravel(),
    cv=5,
    scoring='r2'
)

print("Gradient Boosting Cross-validation R^2 scores:")
print(gbr_cv_scores)
print(f"Average CV R^2: {gbr_cv_scores.mean()}")

# Train model
gbr_model.fit(X_train, y_train.values.ravel())

# Prediction
gbr_pred = gbr_model.predict(X_test)

# Evaluation
gbr_r2 = r2_score(y_test, gbr_pred)
gbr_mae = mean_absolute_error(y_test, gbr_pred)

print(f"Gradient Boosting Test R^2: {gbr_r2}")
print(f"Gradient Boosting Test MAE: {gbr_mae}")

Gradient Boosting Cross-validation R^2 scores:
[0.68114982 0.36131124 0.56024782 0.2005232  0.53936105]
Average CV R^2: 0.46851862532186195
Gradient Boosting Test R^2: 0.2973187576459323
Gradient Boosting Test MAE: 0.7415340583126101


## 长短期记忆

In [34]:
# 长短期记忆模型

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Train-test split
split = int(len(X_aligned) * 0.8)

X_train, X_test = X_aligned.iloc[:split], X_aligned.iloc[split:]
y_train, y_test = y_aligned.iloc[:split], y_aligned.iloc[split:]

# Scale features
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(
    y_train.values.reshape(-1, 1)
)

y_test_scaled = y_scaler.transform(
    y_test.values.reshape(-1, 1)
)

# Reshape for LSTM
# Shape: (samples, timesteps, features)
X_train_lstm = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    1,
    X_train_scaled.shape[1]
)

X_test_lstm = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    1,
    X_test_scaled.shape[1]
)

# Build LSTM model
lstm_model = Sequential()

lstm_model.add(
    LSTM(
        units=64,
        return_sequences=False,
        input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])
    )
)

lstm_model.add(Dropout(0.2))

lstm_model.add(Dense(32, activation='relu'))

lstm_model.add(Dense(1))

# Compile
lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse'
)

# Train
history = lstm_model.fit(
    X_train_lstm,
    y_train_scaled,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Predict
lstm_pred_scaled = lstm_model.predict(X_test_lstm)

# Inverse transform
lstm_pred = y_scaler.inverse_transform(lstm_pred_scaled)

# Evaluation
lstm_r2 = r2_score(y_test, lstm_pred)
lstm_mae = mean_absolute_error(y_test, lstm_pred)

print(f"LSTM Test R^2: {lstm_r2}")
print(f"LSTM Test MAE: {lstm_mae}")

Epoch 1/50
63/63 [==============================] - 4s 16ms/step - loss: 0.0035 - val_loss: 5.8050e-04
Epoch 2/50
63/63 [==============================] - 0s 4ms/step - loss: 0.0011 - val_loss: 5.7913e-04
Epoch 3/50
63/63 [==============================] - 0s 4ms/step - loss: 6.7572e-04 - val_loss: 4.4677e-04
Epoch 4/50
63/63 [==============================] - 0s 4ms/step - loss: 5.2112e-04 - val_loss: 4.9817e-04
Epoch 5/50
63/63 [==============================] - 0s 4ms/step - loss: 4.2413e-04 - val_loss: 3.7112e-04
Epoch 6/50
63/63 [==============================] - 0s 5ms/step - loss: 3.6027e-04 - val_loss: 2.9742e-04
Epoch 7/50
63/63 [==============================] - 0s 4ms/step - loss: 3.5919e-04 - val_loss: 3.5544e-04
Epoch 8/50
63/63 [==============================] - 0s 4ms/step - loss: 3.4131e-04 - val_loss: 2.9710e-04
Epoch 9/50
63/63 [==============================] - 0s 4ms/step - loss: 3.5503e-04 - val_loss: 3.4077e-04
Epoch 10/50
63/63 [==============================] - 

## 预测

In [35]:
# 预测

# Fetch financial statement ratio information
ly_financial_df = rq.get_factor('601318.XSHG', factors, start_date='2026-01-01', end_date='2026-05-01')

ly_financial_df = ly_financial_df.fillna(0, inplace=False)

ly_financial_df = ly_financial_df.groupby([
    pd.Grouper(level='order_book_id'),
    pd.Grouper(level='date', freq='Y')
]).last().fillna(0)

ly_financial_df = ly_financial_df.reorder_levels(['order_book_id', 'date']).shift(-1)

ly_financial_df = ly_financial_df.fillna(0, inplace=False)

print(ly_financial_df.head(20))

                          revenue  operating_revenue  net_operating_revenue  \
order_book_id date                                                            
601318.XSHG   2026-12-31      0.0                0.0                    0.0   

                          book_value_per_share  book_value_per_share_lf  \
order_book_id date                                                        
601318.XSHG   2026-12-31                   0.0                      0.0   

                          book_value_per_share_ttm  return_on_equity  \
order_book_id date                                                     
601318.XSHG   2026-12-31                       0.0               0.0   

                          return_on_asset  profit_before_tax  minority_profit  \
order_book_id date                                                              
601318.XSHG   2026-12-31              0.0                0.0              0.0   

                          ...  long_term_loans  bond_payable  \
order_book_

In [36]:
# Random Forest Prediction
rf_prediction = rf_model.predict(
    selector.transform(ly_financial_df)
)

# Gradient Boosting Regressor Prediction
gbr_prediction = gbr_model.predict(
    ly_financial_df
)

# LSTM Prediction

# Scale
ly_scaled = x_scaler.transform(ly_financial_df)

# Reshape
ly_lstm = ly_scaled.reshape(
    ly_scaled.shape[0],
    1,
    ly_scaled.shape[1]
)

# Predict
lstm_prediction_scaled = lstm_model.predict(ly_lstm)

# Inverse scale
lstm_prediction = y_scaler.inverse_transform(
    lstm_prediction_scaled
)

print(f"Random Forest Prediction: {rf_prediction}")
print(f"Gradient Boosting Regressor Prediction: {gbr_prediction}")
print(f"LSTM Prediction: {lstm_prediction}")

1/1 [==============================] - 0s 32ms/step
Random Forest Prediction: [0.66717431]
Gradient Boosting Regressor Prediction: [0.15021679]
LSTM Prediction: [[-0.40499195]]


## 模型结果

In [37]:
# =========================
# Model Comparison
# =========================

comparison_df = pd.DataFrame({
    'Model': ['Random Forest', 'Gradient Boosting Regression', 'LSTM'],
    'R2': [
        r2,
        gbr_r2,
        lstm_r2
    ],
    'MAE': [
        mae,
        gbr_mae,
        lstm_mae
    ]
})

print(comparison_df)

                          Model        R2       MAE
0                 Random Forest  0.298712  0.752085
1  Gradient Boosting Regression  0.297319  0.741534
2                          LSTM  0.087592  1.045948


## DCF

In [39]:
import numpy as np

def terminal_value_gordon(cf_final_year, discount_rate, growth_rate):
    return cf_final_year * (1 + growth_rate) / (discount_rate - growth_rate)

def dcf_valuation(cash_flows, discount_rate, terminal_value=None):
    """
    Calculate DCF valuation.

    Parameters:
    - cash_flows: list or array of future cash flows [CF1, CF2, ...]
    - discount_rate: required return (e.g. 0.1 for 10%)
    - terminal_value: optional lump sum value at final year

    Returns:
    - present value (DCF valuation)
    """
    cash_flows = np.array(cash_flows, dtype=float)

    # Discount each cash flow
    years = np.arange(1, len(cash_flows) + 1)
    discounted_cf = cash_flows / (1 + discount_rate) ** years

    pv = discounted_cf.sum()

    # Add terminal value if provided
    if terminal_value is not None:
        pv_terminal = terminal_value / (1 + discount_rate) ** len(cash_flows)
        pv += pv_terminal

    return pv

# Assume 15% growth rate and 20% discount rate
growth_rate = 0.15 # 15%
discount_rate = 0.20  # 20%

rf_dcf = dcf_valuation(rf_prediction, discount_rate, terminal_value_gordon(rf_prediction, discount_rate, growth_rate))
gbr_dcf = dcf_valuation(gbr_prediction, discount_rate, terminal_value_gordon(gbr_prediction, discount_rate, growth_rate))
lstm_dcf = dcf_valuation(lstm_prediction, discount_rate, terminal_value_gordon(lstm_prediction, discount_rate, growth_rate))
print("Random Forect DCF Value:", rf_dcf)
print("Gradient Boosting Regressor DCF Value:", gbr_dcf)
print("LSTM DCF Value:", lstm_dcf)

Random Forect DCF Value: [13.34348616]
Gradient Boosting Regressor DCF Value: [3.00433571]
LSTM DCF Value: [[-8.099838]]
